# MLP-VaR Softplus pruebas: downside shocks y memoria de estres - 2010-2024, w=1
 
 Experimento para comprobar si shocks bajistas y memoria reciente de estres reducen el clustering de violaciones sin inflar el VaR medio.
 
 Cambios frente al Softplus base:
 
 - mantiene Softplus en la salida;
 - mantiene `w=1`, ventana rolling de 900 observaciones, seed, normalizacion y funcion de perdida;
 - no usa `vol_5` ni `vol_10` en bruto;
 - anade ratios de volatilidad, shocks absolutos, shocks bajistas y contadores de estres calculados en memoria;
 - evalua todo el periodo `2010-2024` incluido;
 - guarda resumen global, detalle anual, violaciones y comparacion contra el Softplus base `w=1`.
 
 No modifica notebooks antiguos ni sobrescribe CSVs de otros experimentos.

In [1]:
import copy
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import chi2
from torch.utils.data import DataLoader, TensorDataset
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable, *args, **kwargs):
        return iterable

c:\Users\trgglg\OneDrive - gmv.com\projects_w\TFG-GONZALO-LOPEZ-GARCIA-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")
if not DATA_DIR.exists():
    DATA_DIR = Path("data")
DATASET_PATH = DATA_DIR / "dataset_tfg.pkl"

ALPHA = 0.99
SIG = 0.05
W_LOSS = 1
WINDOW = 900
RETRAIN_EVERY = 1
SEED = 42
EVAL_START = "2010-01-01"
EVAL_END = "2024-12-31"

OUT_PRED = DATA_DIR / "nn_softplus_pruebas_downside_stress_2010_2024_w1.csv"
OUT_SUMMARY = DATA_DIR / "softplus_pruebas_downside_stress_2010_2024_w1_summary.csv"
OUT_YEARLY = DATA_DIR / "softplus_pruebas_downside_stress_2010_2024_w1_yearly.csv"
OUT_VIOLATIONS = DATA_DIR / "softplus_pruebas_downside_stress_2010_2024_w1_violations.csv"
OUT_COMPARISON = DATA_DIR / "softplus_pruebas_downside_stress_2010_2024_w1_comparison.csv"

## Carga de datos y nuevas features

In [3]:
dataset = pd.read_pickle(DATASET_PATH).copy().sort_index()
assert isinstance(dataset.index, pd.DatetimeIndex), "El indice debe ser DatetimeIndex"
assert "target" in dataset.columns, "Falta target"
assert "rp_lag_0" in dataset.columns, "Falta rp_lag_0 para calcular shocks"

# Senales relativas: capturan cambio de regimen sin meter volatilidad bruta como nivel permanente.
eps = 1e-8
abs_r = dataset["rp_lag_0"].abs()
downside_r = np.maximum(-dataset["rp_lag_0"], 0.0)
vol_5 = dataset["rp_lag_0"].rolling(5).std(ddof=0)
vol_10 = dataset["rp_lag_0"].rolling(10).std(ddof=0)
vol_20 = dataset["rp_lag_0"].rolling(20).std(ddof=0)

dataset["vol_5_ratio"] = vol_5 / (vol_20 + eps)
dataset["vol_10_ratio"] = vol_10 / (vol_20 + eps)
dataset["shock_1"] = abs_r / (vol_20 + eps)
dataset["shock_5"] = abs_r.rolling(5).max() / (vol_20 + eps)
dataset["downside_shock_1"] = downside_r / (vol_20 + eps)
dataset["downside_shock_5"] = downside_r.rolling(5).max() / (vol_20 + eps)
dataset["downside_sum_5"] = downside_r.rolling(5).sum() / (vol_20 + eps)
dataset["downside_sum_10"] = downside_r.rolling(10).sum() / (vol_20 + eps)
dataset["stress_count_5"] = (dataset["downside_shock_1"] > 1.5).rolling(5).sum()
dataset["stress_count_10"] = (dataset["downside_shock_1"] > 1.5).rolling(10).sum()
dataset["max_downside_10"] = downside_r.rolling(10).max() / (vol_20 + eps)

dataset = dataset.replace([np.inf, -np.inf], np.nan).dropna().copy()

feature_cols = [c for c in dataset.columns if c != "target"]
new_features = [
    "vol_5_ratio", "vol_10_ratio", "shock_1", "shock_5",
    "downside_shock_1", "downside_shock_5", "downside_sum_5", "downside_sum_10",
    "stress_count_5", "stress_count_10", "max_downside_10",
]
print("Dataset:", dataset.shape)
print("Rango:", dataset.index.min().date(), "->", dataset.index.max().date())
print("Features:", len(feature_cols))
print("Nuevas features incluidas:", [c for c in new_features if c in feature_cols])
print("target mean/std/min/max:", dataset["target"].mean(), dataset["target"].std(), dataset["target"].min(), dataset["target"].max())

assert len(feature_cols) == 33, f"Se esperaban 33 features tras anadir downside shocks y memoria de estres, hay {len(feature_cols)}"
assert all(c in feature_cols for c in new_features), "Faltan features nuevas"
assert dataset["target"].abs().quantile(0.999) < 0.2, "El target parece tener escala inesperada"

Dataset: (4912, 34)
Rango: 2005-04-29 -> 2024-12-27
Features: 33
Nuevas features incluidas: ['vol_5_ratio', 'vol_10_ratio', 'shock_1', 'shock_5', 'downside_shock_1', 'downside_shock_5', 'downside_sum_5', 'downside_sum_10', 'stress_count_5', 'stress_count_10', 'max_downside_10']
target mean/std/min/max: -0.0001957865683106607 0.0074329464425199 -0.07361457194412012 0.0784301570334365


## Utilidades de diagnostico

In [4]:
def describe_scale(name, x):
    x = np.asarray(x, dtype=float)
    print(f"\n{name}")
    print("min:", np.nanmin(x))
    print("max:", np.nanmax(x))
    print("mean:", np.nanmean(x))
    print("p95:", np.nanpercentile(x, 95))
    print("p99:", np.nanpercentile(x, 99))
    print("p99.9:", np.nanpercentile(x, 99.9))


def plausibility_report(df, var_col="VaR_pred", loss_col="loss_real"):
    describe_scale("loss_real", df[loss_col].values)
    describe_scale(var_col, df[var_col].values)
    n_negative = int((df[var_col] < 0).sum())
    max_date = df[var_col].idxmax()
    min_date = df[var_col].idxmin()
    print("n_negative_var:", n_negative)
    print("max VaR:", max_date, float(df.loc[max_date, var_col]))
    print("min VaR:", min_date, float(df.loc[min_date, var_col]))

## Metricas de backtesting

In [5]:
def hit_series(real_losses, var_preds):
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds = np.asarray(var_preds, dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def tick_loss(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)
    over = np.maximum(var_preds - real_losses, 0.0)
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def kupiec_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T, x = len(I), int(I.sum())
    p = 1 - alpha
    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}
    p_hat = x / T
    LRuc = -2 * np.log(((1 - p) ** (T - x) * p ** x) / ((1 - p_hat) ** (T - x) * p_hat ** x))
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if pair == (0, 0):
            n00 += 1
        elif pair == (0, 1):
            n01 += 1
        elif pair == (1, 0):
            n10 += 1
        else:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)
        pi11 = n11 / (n10 + n11)
        pi = (n01 + n11) / (n00 + n01 + n10 + n11)
        L0 = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1 = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)
        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]
    if np.isnan(LRuc) or np.isnan(LRind):
        return {"LRind": LRind, "p_ind": p_ind, "LRcc": np.nan, "p_cc": np.nan}
    LRcc = LRuc + LRind
    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": 1 - chi2.cdf(LRcc, df=2)}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha
    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}
    hit = I - p
    y = hit[lags:]
    X = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X
    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}
    beta = np.linalg.inv(XtX) @ X.T @ y
    resid = y - X @ beta
    sigma2 = (resid @ resid) / len(y)
    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}
    DQ = float((beta.T @ XtX @ beta) / sigma2)
    return {"DQ": DQ, "p_dq": 1 - chi2.cdf(DQ, df=X.shape[1])}

In [6]:
def evaluate_var_model(df, alpha=0.99, sig=0.05):
    real_losses = df["loss_real"].values
    var_preds = df["VaR_pred"].values
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    violations = int(I.sum())
    violation_rate = violations / T
    kup = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    df_2020 = df.loc["2020"] if (df.index.year == 2020).any() else pd.DataFrame(columns=df.columns)
    row = {
        "Model": "Softplus + downside shocks + stress memory",
        "w": W_LOSS,
        "T": T,
        "Expected Viol.": (1 - alpha) * T,
        "Violations": violations,
        "Violation Rate": violation_rate,
        "VR Gap": abs(violation_rate - (1 - alpha)),
        "Coverage Bias": violation_rate - (1 - alpha),
        "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
        "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
        "Mean VaR Level": float(np.nanmean(var_preds)),
        "Max VaR Level": float(np.nanmax(var_preds)),
        "Min VaR Level": float(np.nanmin(var_preds)),
        "Mean VaR 2020": float(df_2020["VaR_pred"].mean()) if len(df_2020) else np.nan,
        "Max VaR 2020": float(df_2020["VaR_pred"].max()) if len(df_2020) else np.nan,
        "n_negative_var": int((df["VaR_pred"] < 0).sum()),
        "p_UC": kup["p_uc"],
        "UC Test": "PASS" if kup["p_uc"] > sig else "FAIL",
        "p_CC": cc["p_cc"],
        "CC Test": "PASS" if cc["p_cc"] > sig else "FAIL",
        "p_DQ": dq["p_dq"],
        "DQ Test": "PASS" if dq["p_dq"] > sig else "FAIL",
    }
    vals = [row["p_UC"], row["p_CC"], row["p_DQ"]]
    row["p_Mean"] = float(np.mean([v for v in vals if pd.notnull(v)]))
    row["Valid(CC&DQ)"] = "YES" if row["CC Test"] == "PASS" and row["DQ Test"] == "PASS" else "NO"
    return pd.DataFrame([row])


def evaluate_by_year(df, alpha=0.99):
    rows = []
    for year, g in df.groupby("year"):
        real_losses = g["loss_real"].values
        var_preds = g["VaR_pred"].values
        violations = int(g["viol"].sum())
        expected = (1 - alpha) * len(g)
        kup = kupiec_test(real_losses, var_preds, alpha=alpha)
        cc = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
        dq = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
        rows.append({
            "year": int(year),
            "T": len(g),
            "Expected Viol.": expected,
            "Violations": violations,
            "Violation Rate": violations / len(g),
            "VR Gap": abs((violations / len(g)) - (1 - alpha)),
            "Tick Loss": tick_loss(real_losses, var_preds, alpha=alpha),
            "Winkler Score": winkler_score(real_losses, var_preds, alpha=alpha),
            "Mean VaR Level": float(np.nanmean(var_preds)),
            "Max VaR Level": float(np.nanmax(var_preds)),
            "Min VaR Level": float(np.nanmin(var_preds)),
            "Max Loss": float(np.nanmax(real_losses)),
            "Mean Loss": float(np.nanmean(real_losses)),
            "n_negative_var": int((g["VaR_pred"] < 0).sum()),
            "p_UC": kup["p_uc"],
            "UC Test": "PASS" if kup["p_uc"] > SIG else "FAIL",
            "p_CC": cc["p_cc"],
            "CC Test": "PASS" if cc["p_cc"] > SIG else "FAIL",
            "p_DQ": dq["p_dq"],
            "DQ Test": "PASS" if dq["p_dq"] > SIG else "FAIL",
        })
    return pd.DataFrame(rows)


def violation_table(df):
    cols = ["VaR_pred", "loss_real", "viol", "year"]
    out = df.loc[df["viol"] == 1, cols].copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["exceedance_ratio"] = out["loss_real"] / out["VaR_pred"]
    return out.sort_index()


def worst_days_table(df, n=25):
    out = df.copy()
    out["exceedance"] = out["loss_real"] - out["VaR_pred"]
    out["loss_minus_var"] = out["exceedance"]
    return out.sort_values("loss_real", ascending=False).head(n)

## Modelo Softplus

In [7]:
def inverse_softplus(y):
    return math.log(math.expm1(y))


def var_loss(y_true, y_pred, q=0.99, w=1.0):
    e = y_true - y_pred
    weight = torch.where(e > 0, torch.as_tensor(w, dtype=y_pred.dtype, device=y_pred.device), torch.ones_like(e))
    pinball = torch.maximum(q * e, (q - 1) * e)
    return torch.mean(weight * pinball)


class MLPVaRSoftplus(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(d_in, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )
        self.softplus = nn.Softplus()
        nn.init.constant_(self.body[-1].bias, inverse_softplus(0.02))

    def forward(self, x):
        return self.softplus(self.body(x))


def train_one_model(X_train, y_train, d_in, seed, w_loss, alpha=0.99, max_epochs=200, lr=5e-5, batch_size=256, patience=15, val_split=0.10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    n = len(X_train)
    split = int(n * (1 - val_split))
    X_tr, X_val = X_train[:split], X_train[split:]
    y_tr, y_val = y_train[:split], y_train[split:]
    model = MLPVaRSoftplus(d_in=d_in)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)), batch_size=batch_size, shuffle=False)
    X_val_t = torch.from_numpy(X_val)
    y_val_t = torch.from_numpy(y_val)
    best_loss = np.inf
    best_state = None
    patience_counter = 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad()
            pred = model(xb)
            loss_val = var_loss(yb, pred, q=alpha, w=w_loss)
            loss_val.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = var_loss(y_val_t, val_pred, q=alpha, w=w_loss).item()
        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

## Ejecucion rolling 2010-2024

In [8]:
eval_dates = dataset.loc[pd.Timestamp(EVAL_START):pd.Timestamp(EVAL_END)].index
var_preds = []
real_targets = []
dates = []
current_model = None
muX, sdX = None, None

for i, current_date in enumerate(tqdm(eval_dates, desc="Softplus downside/stress w=1", dynamic_ncols=True)):
    if i % RETRAIN_EVERY == 0:
        train_end = current_date - pd.Timedelta(days=1)
        train_df = dataset.loc[:train_end].tail(WINDOW)
        if len(train_df) < 250:
            if current_model is None:
                continue
        else:
            X_train = train_df[feature_cols].values.astype(np.float32)
            y_train = train_df["target"].values.astype(np.float32).reshape(-1, 1)
            muX = X_train.mean(axis=0, keepdims=True)
            sdX = X_train.std(axis=0, keepdims=True) + 1e-8
            X_train = (X_train - muX) / sdX
            current_model = train_one_model(X_train, y_train, d_in=X_train.shape[1], seed=SEED, w_loss=W_LOSS, alpha=ALPHA)

    if current_model is None or muX is None or sdX is None:
        continue

    test_df = dataset.loc[[current_date]]
    X_test = (test_df[feature_cols].values.astype(np.float32) - muX) / sdX
    y_test = test_df["target"].values.astype(np.float32).reshape(-1, 1)

    current_model.eval()
    with torch.no_grad():
        pred = current_model(torch.from_numpy(X_test)).numpy().reshape(-1)[0]

    var_preds.append(float(pred))
    real_targets.append(float(y_test.reshape(-1)[0]))
    dates.append(current_date)

pred_df = pd.DataFrame({"VaR_pred": var_preds, "loss_real": real_targets}, index=pd.DatetimeIndex(dates)).sort_index()
pred_df = pred_df.loc[EVAL_START:EVAL_END].dropna().copy()
pred_df["viol"] = (pred_df["loss_real"] > pred_df["VaR_pred"]).astype(int)
pred_df["year"] = pred_df.index.year

pred_df.to_csv(OUT_PRED)
print("Guardado:", OUT_PRED)
display(pred_df.head())
display(pred_df.tail())

Softplus downside/stress w=1: 100%|██████████| 3762/3762 [28:19<00:00,  2.21it/s] 


Guardado: ..\data\nn_softplus_pruebas_downside_stress_2010_2024_w1.csv


,VaR_pred,loss_real,viol,year
2010-01-04,0.021194,-0.001401,0,2010
2010-01-05,0.020072,-0.006408,0,2010
2010-01-06,0.019705,0.002147,0,2010
2010-01-07,0.021478,-0.003234,0,2010
2010-01-08,0.021495,-0.003497,0,2010


,VaR_pred,loss_real,viol,year
2024-12-20,0.020419,0.000011,0,2024
2024-12-23,0.020283,-0.004839,0,2024
2024-12-24,0.018824,-0.000006,0,2024
2024-12-26,0.021062,0.001138,0,2024
2024-12-27,0.022615,0.000612,0,2024


## Resultados y comparacion con Softplus base w=1

In [9]:
plausibility_report(pred_df)

summary = evaluate_var_model(pred_df, alpha=ALPHA, sig=SIG)
yearly = evaluate_by_year(pred_df, alpha=ALPHA)
violations = violation_table(pred_df)

summary.to_csv(OUT_SUMMARY, index=False)
yearly.to_csv(OUT_YEARLY, index=False)
violations.to_csv(OUT_VIOLATIONS)

print("Guardado:", OUT_SUMMARY)
print("Guardado:", OUT_YEARLY)
print("Guardado:", OUT_VIOLATIONS)

print("\nResumen global 2010-2024")
display(summary)

print("\nDiagnostico anual completo")
display(yearly)

print("\nAnos ordenados por tasa de violacion")
display(yearly.sort_values(["Violation Rate", "Violations"], ascending=False))

print("\nDetalle COVID 2020")
display(pred_df.loc["2020"].describe())
display(yearly.loc[yearly["year"] == 2020])

print("\nViolaciones completas")
display(violations)

print("\nPeores dias por perdida realizada")
display(worst_days_table(pred_df, n=25))


loss_real
min: -0.07361457496881485
max: 0.07843015342950821
mean: -0.00016187175334081243
p95: 0.009858225146308528
p99: 0.017429363690316665
p99.9: 0.038501730956141275

VaR_pred
min: 0.008057947270572186
max: 0.030873963609337807
mean: 0.02019502433756607
p95: 0.022190690133720634
p99: 0.023304746635258194
p99.9: 0.025987162318080875
n_negative_var: 0
max VaR: 2020-05-07 00:00:00 0.030873963609337807
min VaR: 2020-03-23 00:00:00 0.008057947270572186
Guardado: ..\data\softplus_pruebas_downside_stress_2010_2024_w1_summary.csv
Guardado: ..\data\softplus_pruebas_downside_stress_2010_2024_w1_yearly.csv
Guardado: ..\data\softplus_pruebas_downside_stress_2010_2024_w1_violations.csv

Resumen global 2010-2024


,Model,w,T,Expected Viol.,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,...,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus + downside shocks + stress memory,1,3762,37.62,25,0.006645,0.003355,-0.003355,0.000283,0.020356,...,0.030874,0,0.027651,FAIL,0.032905,FAIL,0.0,FAIL,0.020185,NO



Diagnostico anual completo


,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
0,2010,252,2.52,1,0.003968,0.006032,0.000233,0.020425,0.020374,0.024237,0.017114,0.026197,-0.000381,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
1,2011,252,2.52,3,0.011905,0.001905,0.000256,0.020588,0.020491,0.023446,0.015812,0.026702,-0.000277,0,0.767969,PASS,0.923289,PASS,9.985401e-01,PASS
2,2012,249,2.49,0,0.000000,0.010000,0.000204,0.020247,0.020247,0.023933,0.017454,0.019407,-0.000117,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
3,2013,251,2.51,2,0.007968,0.002032,0.000265,0.020169,0.020038,0.022786,0.016439,0.029816,0.000066,0,0.737311,PASS,0.930176,PASS,9.993244e-01,PASS
4,2014,252,2.52,1,0.003968,0.006032,0.000214,0.019814,0.019773,0.022395,0.017282,0.025232,0.000438,0,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS
5,2015,252,2.52,0,0.000000,0.010000,0.000194,0.019822,0.019822,0.024302,0.015110,0.019679,0.000456,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
6,2016,250,2.50,0,0.000000,0.010000,0.000207,0.020273,0.020273,0.025682,0.014756,0.016855,-0.000410,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
7,2017,249,2.49,0,0.000000,0.010000,0.000210,0.020472,0.020472,0.023079,0.017811,0.014269,-0.000486,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
8,2018,250,2.50,0,0.000000,0.010000,0.000197,0.020045,0.020045,0.022884,0.014961,0.018189,0.000311,0,NaN,FAIL,NaN,FAIL,NaN,FAIL
9,2019,251,2.51,0,0.000000,0.010000,0.000209,0.020264,0.020264,0.024631,0.015424,0.018302,-0.000611,0,NaN,FAIL,NaN,FAIL,NaN,FAIL



Anos ordenados por tasa de violacion


...

,year,T,Expected Viol.,Violations,Violation Rate,VR Gap,Tick Loss,Winkler Score,Mean VaR Level,Max VaR Level,Min VaR Level,Max Loss,Mean Loss,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test
10,2020,251,2.51,12,0.047809,0.037809,0.001155,0.022113,0.020204,0.030874,0.008058,0.07843,-0.000757,0,0.000014,FAIL,0.000067,FAIL,4.889170e-07,FAIL



Violaciones completas


,VaR_pred,loss_real,viol,year,exceedance,exceedance_ratio
2010-02-03,0.019906,0.026197,1,2010,0.006290,1.315991
2011-05-04,0.019805,0.023495,1,2011,0.003691,1.186348
2011-09-21,0.019617,0.026702,1,2011,0.007085,1.361138
2011-12-13,0.022394,0.023733,1,2011,0.001339,1.059799
2013-04-12,0.019393,0.029816,1,2013,0.010423,1.537473
2013-06-19,0.020628,0.026508,1,2013,0.005879,1.285007
2014-11-26,0.020048,0.025232,1,2014,0.005184,1.258599
2020-03-06,0.025165,0.067146,1,2020,0.041981,2.668187
2020-03-10,0.012763,0.023804,1,2020,0.011041,1.865108
2020-03-11,0.015994,0.036496,1,2020,0.020501,2.281815



Peores dias por perdida realizada


,VaR_pred,loss_real,viol,year,exceedance,loss_minus_var
2020-03-17,0.015285,0.078430,1,2020,0.063145,0.063145
2020-03-06,0.025165,0.067146,1,2020,0.041981,0.041981
2020-04-24,0.014265,0.054992,1,2020,0.040727,0.040727
2020-03-13,0.015929,0.044890,1,2020,0.028961,0.028961
2020-03-11,0.015994,0.036496,1,2020,0.020501,0.020501
2020-03-19,0.011433,0.032181,1,2020,0.020748,0.020748
2021-11-24,0.019402,0.030524,1,2021,0.011122,0.011122
2013-04-12,0.019393,0.029816,1,2013,0.010423,0.010423
2022-03-08,0.019722,0.026797,1,2022,0.007076,0.007076
2011-09-21,0.019617,0.026702,1,2011,0.007085,0.007085


In [10]:
# Comparacion directa contra el Softplus base w=1 combinando validacion 2010-2014 y test 2015-2024.
base_paths = [DATA_DIR / "nn_softplus_validation_w1.csv", DATA_DIR / "nn_softplus_test_w1.csv"]
base_parts = []
for path in base_paths:
    if path.exists():
        part = pd.read_csv(path, index_col=0, parse_dates=True)
        base_parts.append(part)
        print("Base cargada:", path, part.index.min().date(), "->", part.index.max().date(), part.shape)

if base_parts:
    base_df = pd.concat(base_parts).sort_index()
    base_df = base_df.loc[EVAL_START:EVAL_END]
    base_df = base_df[~base_df.index.duplicated(keep="last")].copy()
    base_df["viol"] = (base_df["loss_real"] > base_df["VaR_pred"]).astype(int)
    base_df["year"] = base_df.index.year

    base_summary = evaluate_var_model(base_df, alpha=ALPHA, sig=SIG)
    base_summary["Model"] = "Softplus base w=1"
    cmp = pd.concat([base_summary, summary], ignore_index=True)
    cmp.to_csv(OUT_COMPARISON, index=False)
    print("Guardado:", OUT_COMPARISON)

    print("\nComparacion global")
    display(cmp[[
        "Model", "w", "T", "Violations", "Violation Rate", "VR Gap", "Coverage Bias",
        "Tick Loss", "Winkler Score", "Mean VaR Level", "Max VaR Level", "Min VaR Level",
        "Mean VaR 2020", "Max VaR 2020", "n_negative_var", "p_UC", "UC Test", "p_CC", "CC Test", "p_DQ", "DQ Test", "p_Mean", "Valid(CC&DQ)",
    ]])

    base_yearly = evaluate_by_year(base_df, alpha=ALPHA).add_prefix("base_")
    new_yearly = yearly.add_prefix("new_")
    yearly_cmp = base_yearly.merge(new_yearly, left_on="base_year", right_on="new_year", how="outer")
    yearly_cmp["delta_violations_new_minus_base"] = yearly_cmp["new_Violations"] - yearly_cmp["base_Violations"]
    yearly_cmp["delta_rate_new_minus_base"] = yearly_cmp["new_Violation Rate"] - yearly_cmp["base_Violation Rate"]
    yearly_cmp["delta_tick_new_minus_base"] = yearly_cmp["new_Tick Loss"] - yearly_cmp["base_Tick Loss"]
    yearly_cmp["delta_winkler_new_minus_base"] = yearly_cmp["new_Winkler Score"] - yearly_cmp["base_Winkler Score"]

    print("\nComparacion anual base vs nuevo")
    display(yearly_cmp)

    print("\nViolaciones base")
    display(violation_table(base_df))
else:
    print("No existen CSVs base w=1; se omite comparacion contra Softplus base.")

Base cargada: ..\data\nn_softplus_validation_w1.csv 2010-01-04 -> 2014-12-31 (1256, 4)
Base cargada: ..\data\nn_softplus_test_w1.csv 2015-01-02 -> 2024-12-27 (2506, 4)
Guardado: ..\data\softplus_pruebas_downside_stress_2010_2024_w1_comparison.csv

Comparacion global


,Model,w,T,Violations,Violation Rate,VR Gap,Coverage Bias,Tick Loss,Winkler Score,Mean VaR Level,...,Max VaR 2020,n_negative_var,p_UC,UC Test,p_CC,CC Test,p_DQ,DQ Test,p_Mean,Valid(CC&DQ)
0,Softplus base w=1,1,3762,36,0.009569,0.000431,-0.000431,0.000283,0.018417,0.018217,...,0.030089,0,0.789181,PASS,0.016972,FAIL,0.0,FAIL,0.268718,NO
1,Softplus + downside shocks + stress memory,1,3762,25,0.006645,0.003355,-0.003355,0.000283,0.020356,0.020195,...,0.030874,0,0.027651,FAIL,0.032905,FAIL,0.0,FAIL,0.020185,NO



Comparacion anual base vs nuevo


,base_year,base_T,base_Expected Viol.,base_Violations,base_Violation Rate,base_VR Gap,base_Tick Loss,base_Winkler Score,base_Mean VaR Level,base_Max VaR Level,...,new_p_UC,new_UC Test,new_p_CC,new_CC Test,new_p_DQ,new_DQ Test,delta_violations_new_minus_base,delta_rate_new_minus_base,delta_tick_new_minus_base,delta_winkler_new_minus_base
0,2010,252,2.52,1,0.003968,0.006032,0.000221,0.018602,0.018536,0.020590,...,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS,0,0.000000,0.000011,0.001823
1,2011,252,2.52,3,0.011905,0.001905,0.000267,0.018728,0.018568,0.021455,...,0.767969,PASS,0.923289,PASS,9.985401e-01,PASS,0,0.000000,-0.000012,0.001861
2,2012,249,2.49,1,0.004016,0.005984,0.000188,0.018511,0.018508,0.021134,...,NaN,FAIL,NaN,FAIL,NaN,FAIL,-1,-0.004016,0.000016,0.001736
3,2013,251,2.51,2,0.007968,0.002032,0.000262,0.018453,0.018291,0.021049,...,0.737311,PASS,0.930176,PASS,9.993244e-01,PASS,0,0.000000,0.000002,0.001717
4,2014,252,2.52,1,0.003968,0.006032,0.000207,0.018328,0.018271,0.020922,...,0.273177,PASS,0.546423,PASS,8.185397e-01,PASS,0,0.000000,0.000007,0.001486
5,2015,252,2.52,2,0.007937,0.002063,0.000200,0.017786,0.017732,0.022913,...,NaN,FAIL,NaN,FAIL,NaN,FAIL,-2,-0.007937,-0.000006,0.002036
6,2016,250,2.50,0,0.000000,0.010000,0.000183,0.017928,0.017928,0.021035,...,NaN,FAIL,NaN,FAIL,NaN,FAIL,0,0.000000,0.000023,0.002345
7,2017,249,2.49,0,0.000000,0.010000,0.000188,0.018336,0.018336,0.020127,...,NaN,FAIL,NaN,FAIL,NaN,FAIL,0,0.000000,0.000021,0.002136
8,2018,250,2.50,1,0.004000,0.006000,0.000180,0.018108,0.018104,0.020467,...,NaN,FAIL,NaN,FAIL,NaN,FAIL,-1,-0.004000,0.000017,0.001936
9,2019,251,2.51,0,0.000000,0.010000,0.000186,0.018006,0.018006,0.020910,...,NaN,FAIL,NaN,FAIL,NaN,FAIL,0,0.000000,0.000023,0.002259



Violaciones base


,VaR_pred,loss_real,viol,year,exceedance,exceedance_ratio
2010-02-03,0.018052,0.026197,1,2010,0.008145,1.451189
2011-05-04,0.018492,0.023495,1,2011,0.005004,1.270588
2011-09-21,0.018288,0.026702,1,2011,0.008414,1.460081
2011-12-13,0.017262,0.023733,1,2011,0.006471,1.374866
2012-06-20,0.019052,0.019407,1,2012,0.000355,1.018614
2013-04-12,0.017781,0.029816,1,2013,0.012035,1.676834
2013-06-19,0.018407,0.026508,1,2013,0.008100,1.440046
2014-11-26,0.018054,0.025232,1,2014,0.007179,1.397643
2015-08-21,0.014660,0.017541,1,2015,0.002881,1.196512
2015-08-31,0.015723,0.019679,1,2015,0.003955,1.251549


## Lectura esperada
 
 Comparar la fila `Softplus + downside shocks + stress memory` contra `Softplus base w=1` y revisar tambien la tabla anual:
 
 - Objetivo principal: que `UC`, `CC` y `DQ` no fallen globalmente, con especial atencion a `DQ`.
 - La cobertura debe quedar cerca de 1%, no muy por debajo como en el experimento con `vol_5`/`vol_10` en bruto.
 - `Mean VaR Level` y `Max VaR Level` no deberian inflarse claramente frente al Softplus base.
 - En 2020 buscamos menos clustering sin crear un VaR maximo economicamente absurdo.
 - Si `n_negative_var` sigue en cero, Softplus mantiene la restriccion positiva.